# Spotify Customer Churn: Data Preprocessing

## Project Objective

The purpose of this notebook is to prepare the cleaned Spotify customer churn dataset for machine learning model development.

The preprocessing steps include:

- Separating predictor and target variables
- Creating dummy variables for categorical features
- Standardizing numerical features
- Splitting the data into training and testing datasets

The resulting datasets will be used in the modeling phase of the project.

In [1]:
# Import libraries required for preprocessing.

import pandas as pd
import numpy as np

# Load the cleaned Spotify churn dataset.

spotify_df = pd.read_csv("spotify_churn_cleaned.csv")

# Display the first five records.

spotify_df.head()

,user_id,gender,age,country,subscription_type,listening_time,songs_played_per_day,skip_rate,device_type,ads_listened_per_week,offline_listening,is_churned
0,1,Female,54,CA,Free,26,23,0.20,Desktop,31,0,1
1,2,Other,33,DE,Family,141,62,0.34,Web,0,1,0
2,3,Male,38,AU,Premium,199,38,0.04,Mobile,0,1,1
3,4,Female,22,CA,Student,36,2,0.31,Mobile,0,1,0
4,5,Other,29,US,Family,250,57,0.36,Mobile,0,1,1


# Separate Features and Target Variable

The target variable for this project is **is_churned**, where:

- 0 = Customer Retained
- 1 = Customer Churned

Before preprocessing, the target variable will be separated from the predictor variables.

Additionally, the **user_id** field will be removed because it serves only as a unique identifier and does not provide predictive value for machine learning models.

In [2]:
# Create predictor matrix (X) and target variable (y).
# Remove user_id because it is an identifier and not a meaningful feature.

X = spotify_df.drop(['user_id', 'is_churned'], axis=1)

y = spotify_df['is_churned']

# Display dimensions of features and target.

print("Feature Matrix Shape:", X.shape)
print("Target Variable Shape:", y.shape)

Feature Matrix Shape: (8000, 10)
Target Variable Shape: (8000,)


# Create Dummy Variables

Machine learning algorithms require numerical input. Several features in this dataset are categorical and cannot be used directly by most machine learning models.

The following categorical variables will be converted into dummy (one-hot encoded) variables:

- gender
- country
- subscription_type
- device_type

Creating dummy variables allows each category to be represented numerically while preserving the information contained within the original feature.

In [3]:
# Identify categorical variables that require one-hot encoding.

categorical_features = [
    'gender',
    'country',
    'subscription_type',
    'device_type'
]

# Convert categorical variables into dummy variables.

X_encoded = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True
)

# Display the dimensions of the encoded dataset.

print("Original Shape:", X.shape)
print("Encoded Shape:", X_encoded.shape)

Original Shape: (8000, 10)
Encoded Shape: (8000, 20)


## Dummy Variable Findings

The categorical variables were successfully converted into dummy variables using one-hot encoding.

The number of predictor variables increased from 10 to 20 after encoding. This transformation allows machine learning algorithms to process categorical information as numerical inputs while avoiding ordinal relationships between categories.

In [4]:
# Verify that all remaining features are numeric after encoding.

X_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        8000 non-null   int64  
 1   listening_time             8000 non-null   int64  
 2   songs_played_per_day       8000 non-null   int64  
 3   skip_rate                  8000 non-null   float64
 4   ads_listened_per_week      8000 non-null   int64  
 5   offline_listening          8000 non-null   int64  
 6   gender_Male                8000 non-null   bool   
 7   gender_Other               8000 non-null   bool   
 8   country_CA                 8000 non-null   bool   
 9   country_DE                 8000 non-null   bool   
 10  country_FR                 8000 non-null   bool   
 11  country_IN                 8000 non-null   bool   
 12  country_PK                 8000 non-null   bool   
 13  country_UK                 8000 non-null   bool 

# Create Training and Testing Datasets

To evaluate model performance fairly, the dataset will be split into training and testing subsets.

The training dataset will be used to fit machine learning models, while the testing dataset will be reserved for evaluating model performance on unseen data.

An 80/20 split will be used, with stratification applied to preserve the original churn distribution across both datasets.

In [5]:
# Split the data into training and testing datasets.
# Stratification preserves the proportion of churned and retained customers.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)
print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (6400, 20)
Testing Features: (1600, 20)
Training Target: (6400,)
Testing Target: (1600,)


# Feature Standardization

Several numerical variables in this dataset exist on different scales.

For example:

- Age ranges from approximately 16 to 59
- Listening Time ranges from 10 to 299
- Skip Rate ranges from 0 to 0.6

Machine learning algorithms can be sensitive to differences in feature magnitude. Standardization transforms numerical features so they have a mean of 0 and a standard deviation of 1.

To prevent data leakage, the scaler will be fit only on the training dataset and then applied to both the training and testing datasets.

In [6]:
# Standardize numerical features using StandardScaler.
# Fit the scaler on the training data only to prevent data leakage.

from sklearn.preprocessing import StandardScaler

# Identify numerical columns that require scaling.

numeric_features = [
    'age',
    'listening_time',
    'songs_played_per_day',
    'skip_rate',
    'ads_listened_per_week'
]

# Create scaler.

scaler = StandardScaler()

# Fit scaler on training data and transform both datasets.

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(
    X_train[numeric_features]
)

X_test_scaled[numeric_features] = scaler.transform(
    X_test[numeric_features]
)

# Display first few rows of scaled training data.

X_train_scaled.head()

,age,listening_time,songs_played_per_day,skip_rate,ads_listened_per_week,offline_listening,gender_Male,gender_Other,country_CA,country_DE,country_FR,country_IN,country_PK,country_UK,country_US,subscription_type_Free,subscription_type_Premium,subscription_type_Student,device_type_Mobile,device_type_Web
4661,0.810863,-0.187082,-0.421257,1.667137,-0.510744,1,False,True,False,False,False,False,False,False,True,False,False,True,False,False
5195,0.495893,-0.901858,-0.491686,-0.807718,-0.510744,1,True,False,False,False,False,False,False,False,True,False,True,False,False,True
7123,-0.685245,-0.758903,0.881674,1.264253,0.660855,0,True,False,False,False,False,False,True,False,False,True,False,False,False,True
3764,1.440804,-0.461079,-0.315614,-0.577499,-0.510744,1,True,False,False,False,False,False,True,False,False,False,False,True,True,False
6824,-0.449017,-0.580209,-1.160759,-0.750163,-0.510744,1,False,True,False,False,False,False,False,True,False,False,False,False,False,True


In [7]:
# Verify that the scaled numerical variables have approximately
# mean 0 and standard deviation 1.

X_train_scaled[numeric_features].describe()

,age,listening_time,songs_played_per_day,skip_rate,ads_listened_per_week
count,6.400000e+03,6.400000e+03,6.400000e+03,6.400000e+03,6.400000e+03
mean,2.337019e-16,9.769963e-17,-8.215650e-17,-1.776357e-16,3.608225e-17
std,1.000078e+00,1.000078e+00,1.000078e+00,1.000078e+00,1.000078e+00
min,-1.708897e+00,-1.711938e+00,-1.724189e+00,-1.728594e+00,-5.107436e-01
25%,-8.427296e-01,-8.661193e-01,-8.790442e-01,-8.652730e-01,-5.107436e-01
50%,2.343820e-02,-8.387456e-03,1.315037e-03,-1.951466e-03,-5.107436e-01
75%,8.896060e-01,8.612573e-01,8.816743e-01,8.613700e-01,-1.446192e-01
max,1.677031e+00,1.730902e+00,1.726819e+00,1.724692e+00,3.077276e+00


## Standardization Findings

The numerical variables were successfully standardized using StandardScaler.

The transformed features have means approximately equal to 0 and standard deviations approximately equal to 1, confirming that the scaling process was applied correctly.

Standardization ensures that numerical variables with different magnitudes contribute equally during model training and helps improve the performance of many machine learning algorithms.

# Preprocessing Conclusion

The Spotify customer churn dataset was successfully prepared for machine learning model development.

The following preprocessing steps were completed:

- Removed the identifier variable (user_id)
- Separated predictor and target variables
- Converted categorical variables into dummy variables using one-hot encoding
- Split the dataset into training and testing subsets
- Standardized numerical features using StandardScaler
- Validated the scaling results

The resulting datasets are now ready for machine learning model training and evaluation.